# 🐍 파이프라인 실습 (Python)

**커널: Python (pipeline uv 환경)** — 이 노트북은 반드시 아래처럼 띄워야 `kra_predict`가 import 됩니다:

```bash
cd pipeline && uv run --with jupyterlab jupyter lab --notebook-dir=..
```

여기서는 **네트워크 없이(픽스처)** 수집→조립→스코어링 흐름을 한 경주로 따라갑니다.

In [1]:
from kra_predict.api.client import KraClient
from kra_predict.fetch import fetch_meet_bundle, summarize_bundle

# offline=True: pipeline/fixtures만 사용 — API 호출 0회, 텔레메트리도 기록 안 됨
client = KraClient(offline=True)
bundle = fetch_meet_bundle(client, "2026-08-15")
client.close()
print(summarize_bundle(bundle))
print("HTTP 호출:", client.http_calls, "회")

픽스처 없음 → 빈 응답 처리: chulmainfo/numOfRows=100_pageNo=1_race_dt=20260815_rccrs_cd=1.json


픽스처 없음 → 빈 응답 처리: chulmainfo/numOfRows=100_pageNo=1_race_dt=20260815_rccrs_cd=3.json


픽스처 없음 → 빈 응답 처리: textDataHoldBuPtinInfo/numOfRows=100_pageNo=1_race_dt=20260815.json


픽스처 없음 → 빈 응답 처리: Jockey_Change_Detail/meet=3_numOfRows=100_pageNo=1_rc_date=20260815.json


픽스처 없음 → 빈 응답 처리: Jockey_Change_Detail/meet=2_numOfRows=100_pageNo=1_rc_date=20260815.json


...

픽스처 없음 → 빈 응답 처리: textDataHoldSeRaceInfo/numOfRows=100_pageNo=1_race_dt=20260815.json


픽스처 없음 → 빈 응답 처리: textDataHoldBuRaceInfo/numOfRows=100_pageNo=1_race_dt=20260815.json


픽스처 없음 → 빈 응답 처리: Race_Result_total/meet=3_numOfRows=100_pageNo=1_rc_date=20260815.json


픽스처 없음 → 빈 응답 처리: Race_Result_total/meet=2_numOfRows=100_pageNo=1_rc_date=20260815.json


개최일 2026-08-15
  seoul 경주  2 · 출전마  13 · 1년전적  13 · 결과행   7
  busan 경주  0 · 출전마   0 · 1년전적   0 · 결과행   0
  jeju  경주  1 · 출전마   6 · 1년전적   6 · 결과행   0
HTTP 호출: 0 회


In [2]:
# raw 번들 → 데이터 계약(RaceFile) 조립
from kra_predict.features import assemble_races

races = assemble_races(bundle)
race = races[0]
{ "경주": (race["date"], race["track"], race["raceNo"]),
  "출전": len(race["entries"]),
  "첫 엔트리": {k: race["entries"][0][k] for k in ("gateNo", "horseName", "rating", "record1y")} }

{'경주': ('2026-08-15', 'seoul', 1),
 '출전': 7,
 '첫 엔트리': {'gateNo': 1,
  'horseName': '바람의질주',
  'rating': 32.0,
  'record1y': {'starts': 8, 'wins': 1, 'seconds': 2, 'thirds': 1}}}

In [3]:
# 스코어링 — 활성 모델(v2가 있으면 v2)로 예측 생성
from kra_predict.score import build_prediction, model_version

pred = build_prediction(race["entries"], race_date=race["date"], generated_at="lab")
print("활성 모델:", model_version())
print("단승픽:", pred["topPicks"]["win"], "| 신뢰도:", pred["confidence"])
for r in pred["rankings"][:3]:
    print(f"  {r['predictedRank']}위 {r['gateNo']}번 winProb={r['winProb']:.1%}")

활성 모델: v2
단승픽: 2 | 신뢰도: high
  1위 2번 winProb=34.1%
  2위 4번 winProb=16.8%
  3위 1번 winProb=16.2%


In [4]:
# 멱등성 실험 — 타임스탬프만 다르면 '무변경' (cron이 빈 커밋을 안 만드는 이유)
import tempfile
from pathlib import Path
from kra_predict.emit import write_json_if_changed

with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "x.json"
    print("1회차 쓰기:", write_json_if_changed(p, {"updatedAt": "T1", "value": 1}))
    print("타임스탬프만 변경:", write_json_if_changed(p, {"updatedAt": "T2", "value": 1}))
    print("내용 변경:", write_json_if_changed(p, {"updatedAt": "T3", "value": 2}))

1회차 쓰기: True
타임스탬프만 변경: False
내용 변경: True


In [5]:
# ⚠️ 실키 호출 실습 (주석 해제 시 API 쿼터 1회 사용 — pipeline/.env 필요)
# from kra_predict import config
# from kra_predict.api import endpoints as ep
# real = KraClient(config.service_key())
# rows = real.get_items(ep.RACE_PLAN, meet=1, rc_date="20260816")
# real.close()
# print(len(rows), "경주 편성 —", "캐시 히트라면 http_calls=0:", real.http_calls)